# Real Market Data Backtesting with KDB-X Python

This notebook demonstrates how to build a **financial backtesting system** using KDB-X Python. We'll simulate a simple trading strategy and calculate how much profit or loss it would have generated.

## What we explore in this Notebook?


- **Real market data** from Yahoo Finance (last 8 days, minute-level)
- **Moving average crossover strategy** - a classic momentum approach
- **15-minute bars** - aggregated from minute data for cleaner signals
- **Complete trade analysis** - PnL, win rate, drawdowns, and more

## The Strategy: Moving Average Crossover

We'll implement a **dual moving average crossover**:
- **Fast MA**: 10-period moving average (responsive to recent price action)
- **Slow MA**: 30-period moving average (tracks longer-term trend)
- **Buy signal**: When fast MA crosses *above* slow MA → bullish momentum
- **Sell signal**: When fast MA crosses *below* slow MA → bearish momentum

This is a trend-following strategy used by traders to ride momentum waves.

## Prerequisites

- Requires KDB-X to be installed, you can sign up and download on [Developer Center](https://developer.kx.com/products/kdb-x/install). For full install instructions see: [KDB-X Install](https://code.kx.com/kdb-x/).

- To Install [KDB-X Python](https://code.kx.com/pykx/examples/jupyter-integration.html): `pip install --upgrade pykx`


In [4]:
!pip install -qq --upgrade pykx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 7.1 MB/s eta 0:00:00


In [ ]:
import pykx as kx
import yfinance as yf
from datetime import datetime, timedelta

## Step 1: Fetch Real Market Data

We'll download the last 8 days of minute-level data for **AAPL** and **MSFT**. Yahoo Finance provides free intraday data, though it's limited to the most recent period.

### Why minute data?
Minute bars give us fine-grained price action, allowing us to:
- Aggregate into any timeframe we want (5min, 15min, 1hr)
- Simulate realistic entry/exit timing
- See intraday volatility patterns

We'll use Python/yfinance to fetch the data, then convert it to a q table for all subsequent analysis.

In [3]:
# Calculate dynamic date range (last 8 days)
end_date = datetime.now()
start_date = end_date - timedelta(days=8)

print(f"Fetching data from {start_date.date()} to {end_date.date()}...")

# Download minute-level data
data = yf.download(
    tickers=["AAPL", "MSFT"],
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    interval="1m",
    auto_adjust=True
)

# Reshape from wide (columns per ticker) to long format (rows per ticker)
data_long = data.stack(level=1).reset_index()

# Rename columns
data_long = data_long.rename(columns={
    'Datetime': 'dt',
    'Ticker': 'sym'
})

# Convert ticker symbols to lowercase (q convention)
data_long['sym'] = data_long['sym'].str.lower()

# Convert to KDB-X Python table
quotes = kx.toq(data_long)

print(f"✓ Loaded {len(data_long):,} minute bars")
print(f"✓ Tickers: {', '.join(data_long['sym'].unique())}")

Fetching data from 2026-10-11 to 2026-10-19...
✓ Loaded 3,900 minute bars
✓ Tickers: aapl, msft


## Step 2: Inspect the Data Structure

Let's examine what we're working with.

The data should have columns: `dt` (datetime), `sym` (symbol), `Open`, `High`, `Low`, `Close`, `Volume`.

In [4]:
# Show the first 10 quotes
quotes.head(10)

,dt,sym,Close,High,Low,Open,Volume
,,,,,,,
0,2026.10.12D13:30:00.000000000,aapl,340.8,341.47,337.6,338.16,1245538
1,2026.10.12D13:30:00.000000000,msft,539.2,539.54,537.03,538.07,724504
2,2026.10.12D13:31:00.000000000,aapl,341f,341.465,339.76,340.845,294777
3,2026.10.12D13:31:00.000000000,msft,537.055,539.45,536.5,539.06,95894
4,2026.10.12D13:32:00.000000000,aapl,342.1856,342.32,340.71,341.03,361426
5,2026.10.12D13:32:00.000000000,msft,536.51,537.52,535.7013,537.1799,58247
6,2026.10.12D13:33:00.000000000,aapl,341.65,342.3,341.4,342.22,241556
7,2026.10.12D13:33:00.000000000,msft,538.8725,539.0999,536.49,536.49,73845
8,2026.10.12D13:34:00.000000000,aapl,341.74,341.83,341.1,341.65,224156


In [5]:
quotes.dtypes

,columns,datatypes
,,
0,dt,"""kx.TimestampAtom"""
1,sym,"""kx.SymbolAtom"""
2,Close,"""kx.FloatAtom"""
3,High,"""kx.FloatAtom"""
4,Low,"""kx.FloatAtom"""
5,Open,"""kx.FloatAtom"""
6,Volume,"""kx.LongAtom"""


In [6]:
quotes.select(
    columns={
        'bars': kx.Column('sym').count(),
        'start': kx.Column('dt').min(),
        'end': kx.Column('dt').max(),
        'low_price': kx.Column('Close').min(),
        'high_price': kx.Column('Close').max()
        },
        by=kx.Column('sym')
  )

,bars,start,end,low_price,high_price
sym,,,,,
aapl,1950,2026.10.12D13:30:00.000000000,2026.10.16D19:59:00.000000000,339.135,347.836
msft,1950,2026.10.12D13:30:00.000000000,2026.10.16D19:59:00.000000000,524.52,550.985


## Step 3: Aggregate to 15-Minute Bars

Minute data can be noisy. We'll aggregate it into **15-minute OHLCV bars** for cleaner signals.

### The Process:
1. Extract the time component from each datetime
2. Use [`xbar`](https://code.kx.com/pykx/api/columns.html#pykx.wrappers.Column.xbar) to bucket times into 15-minute intervals
3. Aggregate: first Open, max High, min Low, last Close, sum Volume
4. Group by symbol and time bucket

This is standard OHLC bar construction used throughout finance.

In [7]:
# Extract date and time components
quotes = quotes.update(columns={
    'date': kx.Column('dt').date,
    'time': kx.Column('dt').time
})
quotes.dtypes


,columns,datatypes
,,
0,dt,"""kx.TimestampAtom"""
1,sym,"""kx.SymbolAtom"""
2,Close,"""kx.FloatAtom"""
3,High,"""kx.FloatAtom"""
4,Low,"""kx.FloatAtom"""
5,Open,"""kx.FloatAtom"""
6,Volume,"""kx.LongAtom"""
7,date,"""kx.DateAtom"""
8,time,"""kx.TimeAtom"""


In [ ]:
# Create 15-minute time buckets using xbar
# xbar rounds down to the nearest interval (e.g., 14:37 -> 14:30)
quotes = quotes.update(columns={'bucket': kx.Column('time').minute.xbar(15)})

# Aggregate to 15-minute bars
bars = quotes.select(
    columns={
        'open': kx.Column('Open').first(),
        'high': kx.Column('High').max(),
        'low': kx.Column('Low').min(),
        'close': kx.Column('Close').last(),
        'volume': kx.Column('Volume').sum()
    },
    by=[kx.Column('sym'), kx.Column('date'), kx.Column('bucket')]
)

# Recreate datetime column
bars = bars.update(columns={'dt': kx.Column('date') + kx.Column('bucket')})

# Clean up temporary columns
bars = bars.delete(columns=['date', 'bucket'])

# Sort by symbol and datetime for efficiency
bars = kx.q.xasc(['sym', 'dt'], bars)

print("15-minute bars created:")
bars.head(10)

15-minute bars created:


,sym,open,high,low,close,volume,dt
,,,,,,,
0,aapl,338.16,342.32,337.6,341.27,3498896,2026.10.12D13:30:00.000000000
1,aapl,341.27,342.1,340.33,341.05,1286949,2026.10.12D13:45:00.000000000
2,aapl,341.08,342.63,341.016,341.885,1446397,2026.10.12D14:00:00.000000000
3,aapl,341.875,342.11,341.19,341.36,719649,2026.10.12D14:15:00.000000000
4,aapl,341.36,341.579,340.6075,340.835,668248,2026.10.12D14:30:00.000000000
5,aapl,340.8,341.3888,340.78,340.795,547310,2026.10.12D14:45:00.000000000
6,aapl,340.81,341.21,340.5,340.73,525967,2026.10.12D15:00:00.000000000
7,aapl,340.7301,340.84,339.84,340.065,786073,2026.10.12D15:15:00.000000000
8,aapl,340.05,340.055,339.18,339.715,746818,2026.10.12D15:30:00.000000000


## Step 4: Calculate Moving Averages

Now we calculate our strategy indicators:
- **Fast MA**: 10-period moving average using [`mavg`](https://code.kx.com/pykx/api/pykx-execution/q.html#mavg)
- **Slow MA**: 30-period moving average

The `by sym` clause ensures we calculate MAs separately for each ticker - no data leakage between symbols!

A 10-period moving average is the average closing price of the last 10 fifteen-minute bars (fast-moving, reacts quickly to recent price changes), while a 30-period moving average is the average closing price of the last 30 fifteen-minute bars (slow-moving, smooths out short-term fluctuations to show the longer-term trend).

### Identifying Crossovers
A crossover occurs when:
- **Bullish**: Fast MA was ≤ Slow MA, now Fast MA > Slow MA
- **Bearish**: Fast MA was ≥ Slow MA, now Fast MA < Slow MA

We use [`prev`](https://code.kx.com/pykx/api/columns.html#pykx.wrappers.Column.prev) to access the previous row's values for comparison.

In [9]:
# 1. Calculate moving averages per symbol
bars = bars.update(
    columns={
        'ma_fast': kx.Column('close').mavg(10),
        'ma_slow': kx.Column('close').mavg(30)
    },
    by=kx.Column('sym')
)

# 2. Get previous values for crossover detection
bars = bars.update(
    columns={
        'prev_fast': kx.Column('ma_fast').prev(),
        'prev_slow': kx.Column('ma_slow').prev()
    },
    by=kx.Column('sym')
)

# 3. Identify crossover signals
# signal = 1: Bullish Cross
# signal = -1: Bearish Cross
# signal = 0: No Trade/Hold (default for every bar)
bars = bars.update(columns={'signal': kx.IntAtom(0)})

bars = bars.update(
    columns={'signal': kx.IntAtom(1)},
    where=(kx.Column('ma_fast') > kx.Column('ma_slow')) & (kx.Column('prev_fast') <= kx.Column('prev_slow'))
)

bars = bars.update(
    columns={'signal': kx.IntAtom(-1)},
    where=(kx.Column('ma_fast') < kx.Column('ma_slow')) & (kx.Column('prev_fast') >= kx.Column('prev_slow'))
)

print("Trading signals generated:")

# Filter and display rows where signal is not 0
signals = bars.select(where=kx.Column('signal') != 0)
signals.head(10)


Trading signals generated:


,sym,open,high,low,close,volume,dt,ma_fast,ma_slow,prev_fast,prev_slow,signal
,,,,,,,,,,,,
0,aapl,339.81,340.44,339.8,340.35,447671,2026.10.12D16:00:00.000000000,340.6615,340.7168,340.7535,340.7535,-1i
1,aapl,342.135,345.29,341.12,345.23,2212024,2026.10.13D13:45:00.000000000,340.8143,340.4803,340.2313,340.3044,1i
2,aapl,342.2,342.6,341.7,342.4756,647871,2026.10.13D19:00:00.000000000,342.8701,343.1758,343.094,343.0773,-1i
3,aapl,345.24,346.07,344.76,345.855,1162850,2026.10.14D15:00:00.000000000,344.284,344.1297,343.9461,344.1212,1i
4,aapl,341.93,342.91,341.72,342.855,855752,2026.10.14D18:00:00.000000000,343.5695,343.737,343.88,343.7595,-1i
5,aapl,344.67,345.3,344.08,344.78,1104520,2026.10.15D14:45:00.000000000,344.3655,344.2487,344.1945,344.2727,1i
6,aapl,341.9745,342.7,341.9,342.335,565002,2026.10.15D18:30:00.000000000,343.617,343.7025,343.7125,343.6953,-1i
7,aapl,345.2599,346.62,345.21,345.9095,1598804,2026.10.16D15:00:00.000000000,344.062,343.8017,343.766,343.7786,1i
8,aapl,342.52,342.8199,341.11,341.795,1088898,2026.10.16D18:15:00.000000000,343.5606,343.7395,343.8976,343.8281,-1i


## Step 5: Generate Trade Pairs

We have signals, now we need to pair them up into actual trades:
- Each **BUY signal** (signal=1) is an entry
- Each **SELL signal** (signal=-1) is an exit
- We match each entry with the next exit using an asof join

### Why Asof Join?
The [`aj`](https://code.kx.com/q/ref/aj/) function finds the "most recent match as of a time" - perfect for matching "next exit after this entry".

For speed, `aj` needs the table it searches (the buys) sorted by `sym` and time, with the [grouped attribute](https://code.kx.com/q/ref/aj/#performance) (`` `g# ``) applied to `sym`. We apply it with `kx.Column('sym').call('`g#')`.

The same join is also available through the pandas style [`merge_asof`](https://code.kx.com/pykx/user-guide/advanced/Pandas_API.html#tablemerge_asof) method, which calls `aj` underneath.

We'll trade 100 shares per position for simplicity.

In [ ]:
# 1. Extract and label trades
trades = bars.select(
    columns=['sym', 'dt', 'signal', 'close'],
    where=kx.Column('signal') != 0
)

# 2. Separate buys and sells into their own tables
buys = trades.select(
    columns={'sym': kx.Column('sym'), 'entry_dt': kx.Column('dt'), 'entry_price': kx.Column('close')},
    where=kx.Column('signal') == 1
)

sells = trades.select(
    columns={'sym': kx.Column('sym'), 'exit_dt': kx.Column('dt'), 'exit_price': kx.Column('close')},
    where=kx.Column('signal') == -1
)

# 3. Prepare the tables for the asof join
# aj matches on the same column names in both tables, so give buys an exit_dt column holding its entry time
buys = buys.update(columns={'exit_dt': kx.Column('entry_dt')})

# aj is fastest when the table being searched (buys) is sorted by sym then time and has the grouped attribute on sym
buys = kx.q.xasc(['sym', 'exit_dt'], buys)
buys = buys.update(kx.Column('sym').call('`g#'))
sells = kx.q.xasc(['sym', 'exit_dt'], sells)

# 4. Match each SELL with its preceding BUY using an asof join
positions = kx.q.aj(['sym', 'exit_dt'], sells, buys)

# The pandas style merge_asof method performs the same join
# and gives the same result as the aj call above
positions_alt = sells.merge_asof(buys, on='exit_dt', by='sym')

# Reorder columns for readability (entry, then exit)
positions = positions.select(
    columns=['sym', 'entry_dt', 'entry_price', 'exit_dt', 'exit_price']
)

# 5. Clean up - remove sells without a matching buy
positions = positions.select(where=~kx.Column('entry_price').null())
positions = positions.update(columns={'size': kx.IntAtom(100)})

print("Generated positions:")
positions.iloc[:10]

Generated positions:


,sym,entry_dt,entry_price,exit_dt,exit_price,size
,,,,,,
0,aapl,2026.10.13D13:45:00.000000000,345.23,2026.10.13D19:00:00.000000000,342.4756,100i
1,aapl,2026.10.14D15:00:00.000000000,345.855,2026.10.14D18:00:00.000000000,342.855,100i
2,aapl,2026.10.15D14:45:00.000000000,344.78,2026.10.15D18:30:00.000000000,342.335,100i
3,aapl,2026.10.16D15:00:00.000000000,345.9095,2026.10.16D18:15:00.000000000,341.795,100i
4,msft,2026.10.12D16:00:00.000000000,541.715,2026.10.12D18:15:00.000000000,539.2902,100i
5,msft,2026.10.13D13:30:00.000000000,549.41,2026.10.14D13:30:00.000000000,543.18,100i
6,msft,2026.10.15D15:30:00.000000000,537.205,2026.10.16D14:45:00.000000000,534.48,100i


## Step 6: Calculate PnL

For each position, we calculate:
- **PnL in dollars**: `size × (exit_price - entry_price)`
- **PnL percentage**: `100 × (exit_price - entry_price) / entry_price`
- **Holding period**: How long we held the position

This tells us which trades were winners and losers.

In [11]:
# Calculate PnL metrics
positions = positions.update(
    columns={
        'pnl': kx.Column('size') * (kx.Column('exit_price') - kx.Column('entry_price')),              # Dollar P&L
        'pnl_pct': 100 * ((kx.Column('exit_price') - kx.Column('entry_price')) / kx.Column('entry_price')),   # Percentage return
        'holding_period': kx.Column('exit_dt') - kx.Column('entry_dt')  # Time held
    }
)

print("Position PnL:")
positions.iloc[:15]

Position PnL:


,sym,entry_dt,entry_price,exit_dt,exit_price,size,pnl,pnl_pct,holding_period
,,,,,,,,,
0,aapl,2026.10.13D13:45:00.000000000,345.23,2026.10.13D19:00:00.000000000,342.4756,100i,-275.4425,-0.7978521,0D05:15:00.000000000
1,aapl,2026.10.14D15:00:00.000000000,345.855,2026.10.14D18:00:00.000000000,342.855,100i,-300f,-0.8674155,0D03:00:00.000000000
2,aapl,2026.10.15D14:45:00.000000000,344.78,2026.10.15D18:30:00.000000000,342.335,100i,-244.5007,-0.70915,0D03:45:00.000000000
3,aapl,2026.10.16D15:00:00.000000000,345.9095,2026.10.16D18:15:00.000000000,341.795,100i,-411.4471,-1.189465,0D03:15:00.000000000
4,msft,2026.10.12D16:00:00.000000000,541.715,2026.10.12D18:15:00.000000000,539.2902,100i,-242.4805,-0.4476163,0D02:15:00.000000000
5,msft,2026.10.13D13:30:00.000000000,549.41,2026.10.14D13:30:00.000000000,543.18,100i,-622.998,-1.13394,1D00:00:00.000000000
6,msft,2026.10.15D15:30:00.000000000,537.205,2026.10.16D14:45:00.000000000,534.48,100i,-272.5037,-0.5072619,0D23:15:00.000000000


## Step 7: Performance by Symbol

Different stocks behave differently. Let's break down performance by ticker to see which one the strategy works better on.

We might find:
- One stock trends more consistently (better for MA crossover)
- One stock is choppier (more whipsaws, lower win rate)
- Differences in volatility and trade frequency

In [12]:
print(f"\n{'='*60}")
print("OVERALL PERFORMANCE")
print(f"{'='*60}")

total_pnl = kx.q.sum(positions['pnl'])
num_winners = len(positions.select(where=kx.Column('pnl') > 0))
num_losers = len(positions.select(where=kx.Column('pnl') < 0))
total_trades = len(positions)

print(f"Total P&L: ${total_pnl}")
print(f"Total Trades: {total_trades}")
print(f"Winners: {num_winners} ({100*num_winners/total_trades:.1f}%) | Losers: {num_losers} ({100*num_losers/total_trades:.1f}%)")
print(f"Win Rate: {100*num_winners/total_trades:.2f}%")

# Per-symbol breakdown
print(f"\n{'='*60}")
print("PERFORMANCE BY SYMBOL")
print(f"{'='*60}")

# Calculate per-symbol statistics
symbol_stats = positions.select(
    columns={
        'trades': kx.Column('sym').count(),
        'winners': (kx.Column('pnl') > 0).sum(),
        'total_pnl': kx.Column('pnl').sum(),
        'avg_pnl': kx.Column('pnl').avg(),
        'win_rate': 100 * (kx.Column('pnl') > 0).sum() / kx.Column('sym').count(),
        'best_trade': kx.Column('pnl').max(),
        'worst_trade': kx.Column('pnl').min()
    },
    by=kx.Column('sym')
)
print(symbol_stats)

print(f"{'='*60}")



OVERALL PERFORMANCE
Total P&L: $-2369.373
Total Trades: 7
Winners: 0 (0.0%) | Losers: 7 (100.0%)
Win Rate: 0.00%

PERFORMANCE BY SYMBOL
sym | trades winners total_pnl avg_pnl   win_rate best_trade worst_trade
----| ------------------------------------------------------------------
aapl| 4      0       -1231.39  -307.8476 0        -244.5007  -411.4471  
msft| 3      0       -1137.982 -379.3274 0        -242.4805  -622.998   


## Step 8: Cumulative PnL Over Time

To understand the strategy's progression, we calculate cumulative PnL - how our account balance would have grown (or shrunk) over the backtest period.

The [`sums`](https://code.kx.com/pykx/api/columns.html#pykx.wrappers.Column.sums) function creates a running total, perfect for equity curves.

In [13]:
# Sort trades chronologically and calculate running PnL
pnl_timeline = positions.select(
    columns=['entry_dt', 'sym', 'pnl']
)

# Sort by entry_dt
pnl_timeline = kx.q.xasc(['entry_dt'], pnl_timeline)

# Calculate cumulative PnL
pnl_timeline = pnl_timeline.update(
    columns={'cumulative_pnl': kx.Column('pnl').sums()}
)

print("CUMULATIVE PnL OVER TIME:")
pnl_timeline.iloc[-15:]  # Last 15 rows

CUMULATIVE PnL OVER TIME:


,entry_dt,sym,pnl,cumulative_pnl
,,,,
0,2026.10.12D16:00:00.000000000,msft,-242.4805,-242.4805
1,2026.10.13D13:30:00.000000000,msft,-622.998,-865.4785
2,2026.10.13D13:45:00.000000000,aapl,-275.4425,-1140.921
3,2026.10.14D15:00:00.000000000,aapl,-300f,-1440.921
4,2026.10.15D14:45:00.000000000,aapl,-244.5007,-1685.422
5,2026.10.15D15:30:00.000000000,msft,-272.5037,-1957.925
6,2026.10.16D15:00:00.000000000,aapl,-411.4471,-2369.373


## Step 9: Maximum Drawdown Analysis

**Maximum drawdown** is the largest peak-to-trough decline in cumulative PnL. It measures the strategy's worst-case risk.

### Why it matters:
- Shows how much capital you could have lost at the worst point
- Helps determine position sizing and risk tolerance
- More realistic than just looking at final PnL

We calculate:
1. Running maximum PnL at each point (the peak)
2. Current drawdown = Current PnL - Running maximum
3. Maximum drawdown = Largest negative value

In [14]:
# Calculate running maximum (the peak)
pnl_timeline = pnl_timeline.update(
    columns={'running_max': kx.Column('cumulative_pnl').maxs()}
)

# Calculate drawdown from peak
pnl_timeline = pnl_timeline.update(
    columns={'drawdown': kx.Column('cumulative_pnl') - kx.Column('running_max')}
)

# Find maximum drawdown
max_dd = pnl_timeline['drawdown'].min()
print(f"Maximum Drawdown: ${max_dd}")

# Show when it occurred
print("\nWorst drawdown period:")
worst_drawdown = pnl_timeline.select(
    where=kx.Column('drawdown') == kx.Column('drawdown').min()
)
worst_drawdown

Maximum Drawdown: $-2126.892

Worst drawdown period:


,entry_dt,sym,pnl,cumulative_pnl,running_max,drawdown
,,,,,,
0,2026.10.16D15:00:00.000000000,aapl,-411.4471,-2369.373,-242.4805,-2126.892


## Step 10: Testing Different Parameters

What if we used **different MA periods**? Let's try a faster 5/20 combination and compare signal frequency.

### Trade-offs:
- **Shorter MAs** (5/20): More signals, more responsive, but more whipsaws
- **Longer MAs** (10/30): Fewer signals, smoother, but slower to react

This is called **parameter sensitivity analysis** - testing how results change with different settings.

In [15]:
# Calculate alternative moving averages (5/20)
bars2 = bars.update(
    columns={
        'ma_fast5': kx.Column('close').mavg(5),
        'ma_slow20': kx.Column('close').mavg(20)
    },
    by=kx.Column('sym')
)

# Detect crossovers with new parameters
# To detect a "crossover," we need to know the values from the PREVIOUS row.
# 'prev' shifts the data down by one, allowing us to compare 'now' vs 'then'.
bars2 = bars2.update(
    columns={
        'prev_fast5': kx.Column('ma_fast5').prev(),
        'prev_slow20': kx.Column('ma_slow20').prev()
    },
    by=kx.Column('sym')
)

# Initialize all signals to 0 (No Trade/Hold)
bars2 = bars2.update(columns={'signal2': kx.IntAtom(0)})

# BULLISH CROSSOVER:
# Fast MA is currently ABOVE slow MA, but was BELOW or EQUAL in the previous bar.
bars2 = bars2.update(
    columns={'signal2': kx.IntAtom(1)},
    where=(kx.Column('ma_fast5') > kx.Column('ma_slow20')) & (kx.Column('prev_fast5') <= kx.Column('prev_slow20'))
)

# BEARISH CROSSOVER:
# Fast MA is currently BELOW slow MA, but was ABOVE or EQUAL in the previous bar
bars2 = bars2.update(
    columns={'signal2': kx.IntAtom(-1)},
    where=(kx.Column('ma_fast5') < kx.Column('ma_slow20')) & (kx.Column('prev_fast5') >= kx.Column('prev_slow20'))
)

# Compare signal counts
signal_count_10_30 = bars.select(kx.Column('signal').abs().sum())['signal'][0]
signal_count_5_20 = bars2.select(kx.Column('signal2').abs().sum())['signal2'][0]

signal_count = kx.Table(data={
    'strategy': ['MA_10_30', 'MA_5_20'],
    'signals': [signal_count_10_30, signal_count_5_20]
})

print("SIGNAL FREQUENCY COMPARISON:")
signal_count

SIGNAL FREQUENCY COMPARISON:


,strategy,signals
,,
0,MA_10_30,15i
1,MA_5_20,18i


#### Challenge: Can you find the PnL stats for this new 5/20 Moving Average strategy?

In [16]:
# Challenge Code:

## Conclusion and Next Steps

### What We Built

This notebook demonstrated:
1. **Real data integration**: Yahoo Finance → KDB-X Python pipeline
2. **Bar aggregation**: Minute data → 15-minute OHLC bars
3. **Technical indicators**: Moving averages with `mavg`
4. **Signal generation**: Crossover detection logic
5. **Trade matching**: Entry/exit pairing with `aj`
6. **Performance analysis**: PnL, win rate, drawdown, cumulative returns
7. **Parameter testing**: Comparing different MA periods

### Key KDB-X Python Features Used

- **`xbar`**: Time bucketing for bar aggregation
- **`mavg`**: Efficient moving averages
- **`by` clause**: Per-symbol calculations without loops
- **`aj` (asof join)** and **`merge_asof`**: Time-series matching for trade pairing
- **`sums` / `maxs`**: Running calculations for equity curves
- **`prev`**: Access previous row values for crossover detection
- **Attributes (`` `g# ``)**: Grouped attribute on `sym` to speed up `aj` lookups

### Improvements for Production

To make this a real trading strategy, add:

1. **Transaction costs**: Commissions ($0.005/share?), slippage (1-2 bps?)
2. **Position sizing**: Risk-based sizing, not fixed 100 shares
3. **Risk management**: Stop losses, maximum position limits
4. **Multiple timeframes**: Confirm signals on higher timeframes
5. **Volume filters**: Ignore signals on low-volume bars
6. **Market regime detection**: Different parameters for trending vs ranging markets
7. **Walk-forward testing**: Test on rolling windows to avoid overfitting
8. **Realistic execution**: Model market impact, time-in-force

### Why KDB-X Python?

This example used just 8 days of minute data. In production:
- Backtests run on **years of tick data** (billions of records)
- Real-time strategies process **millions of quotes per second**
- Portfolio analytics aggregate **thousands of instruments simultaneously**

KDB-X Python excels at these scales with the same clean syntax we used here. The code patterns are identical whether you're processing 10,000 rows or 10 billion.

### Resources

- [KDB-X Python Docs](https://code.kx.com/pykx/index.html)
- [Time-series joins (aj, asof)](https://code.kx.com/pykx/user-guide/advanced/Pandas_API.html#tablemerge_asof)
- [Moving averages (mavg)](https://code.kx.com/pykx/api/columns.html#pykx.wrappers.Column.mavg)
- [KDB-X Python Documentation](https://code.kx.com/pykx/)

Happy backtesting! 🚀